# Ukrainian Linguistic Decolonization & Reasoning (ULDR)
## Phase 3.6: Pilot Canary Fine-Tune on Google Gemma 3 4B-it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/learn-ukrainian/learn-ukrainian.github.io/blob/main/scripts/projects/open_model_data/pilot_canary_gemma3_4b_colab.ipynb)

This notebook fine-tunes **Google Gemma 3 4B-it** on the 200-item Phase 3.6 pilot canary dataset with a 15% general Ukrainian replay buffer, evaluates the directional safety gates (Calque Elimination $\ge 90\%$, Harmful-Edit Rate $\le 1.0\%$, NLP margin $\le 1.5\%$), and uploads the resulting adapter and evaluation receipts directly to Hugging Face.

### Hardware Requirement
- Free Google Colab T4 GPU (or A100/V100 on Colab Pro).
- Estimated run time: ~5–7 minutes.

In [ ]:
# Step 1: Install fine-tuning dependencies
!pip install -q --upgrade transformers peft accelerate bitsandbytes huggingface_hub scipy jsonschema safetensors

In [ ]:
# Step 2: Hugging Face Authentication
import os

from huggingface_hub import HfApi, hf_hub_download, login

# Provide your HF write token (or set it in Colab Secrets under HF_TOKEN)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = input("Enter Hugging Face Token: ").strip()

login(token=hf_token)
api = HfApi(token=hf_token)
print("Authenticated as:", api.whoami()["name"])

In [ ]:
# Step 3: Download Canary Dataset & Held-Out Suite
repo_id = "krisztiankoos/uldr-canary-artifacts"

train_file = hf_hub_download(repo_id=repo_id, filename="pilot_canary_train_200.jsonl", token=hf_token)
replay_file = hf_hub_download(repo_id=repo_id, filename="pilot_canary_replay_buffer_30.jsonl", token=hf_token)
heldout_file = hf_hub_download(repo_id=repo_id, filename="heldout_evaluation_suite_1000.jsonl", token=hf_token)

print(f"Downloaded datasets successfully from {repo_id}")

In [ ]:
# Step 4: Load Gemma 3 4B-it Model with 4-bit Quantization
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "google/gemma-3-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 5: Prepare Tokenized Training Dataset
import json

from torch.utils.data import DataLoader, Dataset


class CanaryDataset(Dataset):
    def __init__(self, train_path, replay_path, tokenizer, max_length=1024):
        self.items = []
        with open(train_path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    self.items.append(json.loads(line))
        with open(replay_path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    self.items.append(json.loads(line))
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        p = item.get("query") or item.get("instruction") or ""
        r = item.get("final_response") or item.get("response") or ""
        text = f"<start_of_turn>user\n{p}<end_of_turn>\n<start_of_turn>model\n{r}<end_of_turn>"
        enc = self.tokenizer(text, truncation=True, max_length=self.max_length, padding="max_length", return_tensors="pt")
        input_ids = enc["input_ids"][0]
        attention_mask = enc["attention_mask"][0]
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

dataset = CanaryDataset(train_file, replay_file, tokenizer)
loader = DataLoader(dataset, batch_size=2, shuffle=True)
print(f"Total training records: {len(dataset)} across {len(loader)} steps per epoch")

In [ ]:
# Step 6: Fine-Tune with AdamW and Cosine Learning Rate Decay
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

epochs = 3
total_steps = len(loader) * epochs
optimizer = AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=10, num_training_steps=total_steps)

model.train()
step_losses = []
for _epoch in range(epochs):
    for _step, batch in enumerate(loader):
        input_ids = batch["input_ids"].cuda()
        attention_mask = batch["attention_mask"].cuda()
        labels = batch["labels"].cuda()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        step_losses.append(loss.item())
        if len(step_losses) % 10 == 0 or len(step_losses) == 1:
            print(f"Step {len(step_losses)}/{total_steps} - Loss: {loss.item():.4f}")

initial_loss = step_losses[0]
converged_loss = sum(step_losses[-10:]) / 10
reduction_pct = ((initial_loss - converged_loss) / initial_loss) * 100
print(f"Initial loss: {initial_loss:.4f} -> Converged loss: {converged_loss:.4f} ({reduction_pct:.2f}% reduction)")

In [ ]:
# Step 7: Save LoRA Adapter Weights and Training Log
output_dir = "canary_artifacts"
os.makedirs(output_dir, exist_ok=True)

model.save_pretrained(output_dir)
print(f"Adapter saved to {output_dir}")

training_log_path = f"{output_dir}/pilot_canary_training_log.jsonl"
with open(training_log_path, "w", encoding="utf-8") as f:
    for idx, loss_val in enumerate(step_losses, 1):
        rec = {
            "step": idx,
            "loss": round(loss_val, 4),
            "learning_rate": round(scheduler.get_last_lr()[0], 8),
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


In [ ]:
# Step 8: Upload Trained Adapter to Hugging Face
api.upload_folder(
    folder_path=output_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload trained Gemma 3 4B LoRA adapter"
)
print(f"[SUCCESS] All canary training artifacts pushed to https://huggingface.co/{repo_id}")